# Day 09. Exercise 02
# Metrics

## 0. Imports

In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [56]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_with_target = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = df_with_target['dayofweek']

In [57]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [59]:
svc = SVC(random_state=21,
          kernel='linear',
          gamma='auto',
          class_weight=None,
          probability=True)
svc.fit(X_train, y_train)
pred = svc.predict(X_test)
pred_proba = svc.predict_proba(X_test)

In [60]:
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')
roc_auc = roc_auc_score(y_test, pred_proba, multi_class='ovr')

In [61]:
print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')
print(f'roc_auc is {roc_auc:.5}')

accuracy is 0.71893
precision is 0.72714
recall is: 0.71893
roc_auc is 0.91142


## 3. Decision tree

1. The same task for decision tree

In [62]:
dt = DecisionTreeClassifier(random_state=21,
                            class_weight=None,
                            criterion='gini',
                            max_depth=31,
                            )
dt.fit(X_train, y_train)
pred = dt.predict(X_test)
pred_proba = dt.predict_proba(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')
roc_auc = roc_auc_score(y_test, pred_proba, multi_class='ovr')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')
print(f'roc_auc is {roc_auc:.5}')

accuracy is 0.86686
precision is 0.87097
recall is: 0.86686
roc_auc is 0.91579


## 4. Random forest

1. The same task for random forest.

In [63]:
rf = RandomForestClassifier(random_state=21,
                            n_estimators=100,
                            max_depth=31, 
                            class_weight=None,
                            criterion='gini')
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')
roc_auc = roc_auc_score(y_test, pred_proba, multi_class='ovr')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')
print(f'roc_auc is {roc_auc:.5}')

accuracy is 0.93787
precision is 0.9391
recall is: 0.93787
roc_auc is 0.98772


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [64]:
rf = RandomForestClassifier(random_state=21,
                            n_estimators=100,
                            max_depth=31, 
                            class_weight=None,
                            criterion='gini')
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)

In [65]:
results = pd.DataFrame({'y_test': y_test, 'prediction': pred})
results['is_error'] = (results['y_test'] != results['prediction']).astype(int)

In [66]:
results[results['is_error'] == 1].groupby('y_test').size()

y_test
0    7
1    3
2    2
3    2
4    3
5    3
6    1
dtype: int64

My model makes the most errors for the day number 0.
This is Monday.

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [67]:
def count_metrics(models, params):
    results = {}
    
    for model, model_params in zip(models, params):
        curr_model = model(**model_params, random_state=21)
        
        curr_model.fit(X_train, y_train)
        pred = curr_model.predict(X_test)
        pred_proba = curr_model.predict_proba(X_test)

        # Вычисление метрик
        metrics = {
            'accuracy': accuracy_score(y_test, pred),
            'precision': precision_score(y_test, pred, average='weighted'),
            'recall': recall_score(y_test, pred, average='weighted'),
            'roc_auc': roc_auc_score(y_test, pred_proba, multi_class='ovr')
        }
        
        results[model.__name__] = metrics
    
    return results

In [68]:
# Пример использования
models = [RandomForestClassifier, SVC]
params = [
    {'n_estimators': 100, 'max_depth': 31},
    {'kernel': 'linear', 'gamma': 'auto', 'probability': True}
]

metrics_results = count_metrics(models, params)

In [69]:
metrics_results

{'RandomForestClassifier': {'accuracy': 0.9378698224852071,
  'precision': 0.9390993534287375,
  'recall': 0.9378698224852071,
  'roc_auc': 0.9877207268160056},
 'SVC': {'accuracy': 0.7189349112426036,
  'precision': 0.7271369623885529,
  'recall': 0.7189349112426036,
  'roc_auc': 0.9114244836329188}}